# DataFrame Relational Operations

Real data is spread over several tables. A bakehouse transaction only stores
a `franchiseID`, and the franchise's name and city live in another table.
This notebook covers the operations that combine DataFrames: **joins**, which
match rows from two DataFrames side by side, and **set operations**, which
stack rows on top of each other. It builds on the single-DataFrame tools of
{doc}`DataFrame API Basics <01_dataframe_api_basics>` and the aggregations of
{doc}`Grouping and Aggregating Data <02_grouping_and_aggregating_data>`.

In [1]:
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

transactions = spark.read.table("samples.bakehouse.sales_transactions")
franchises = spark.read.table("samples.bakehouse.sales_franchises")
customers = spark.read.table("samples.bakehouse.sales_customers")
suppliers = spark.read.table("samples.bakehouse.sales_suppliers")

## A First Join

`df.join(other, on, how)` combines two DataFrames. When the key column has
the same name on both sides, pass that name (or a list of names) as `on`.
Spark then performs an equi-join on it and keeps a single copy of the key
column in the result
([Apache Spark API reference: DataFrame.join](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)):

In [2]:
sales_with_store = transactions.join(franchises, on="franchiseID", how="inner")

sales_with_store.select("transactionID", "franchiseID", "name", "city", "country", "totalPrice").show(5)

+-------------+-----------+----------------+---------+-------+----------+
|transactionID|franchiseID|            name|     city|country|totalPrice|
+-------------+-----------+----------------+---------+-------+----------+
|      1002961|    3000047|Sweet Sinsations|Stockholm| Sweden|        24|
|      1003007|    3000047|Sweet Sinsations|Stockholm| Sweden|       108|
|      1003017|    3000047|Sweet Sinsations|Stockholm| Sweden|       120|
|      1003068|    3000047|Sweet Sinsations|Stockholm| Sweden|        84|
|      1003103|    3000047|Sweet Sinsations|Stockholm| Sweden|        84|
+-------------+-----------+----------------+---------+-------+----------+
only showing top 5 rows


With the franchise attributes attached, revenue per country is one
aggregation away:

In [3]:
(
    sales_with_store
    .groupBy("country")
    .agg(
        F.count_distinct("franchiseID").alias("franchises"),
        F.sum("totalPrice").alias("revenue"),
    )
    .orderBy(F.desc("revenue"))
    .show()
)

+-----------+----------+-------+
|    country|franchises|revenue|
+-----------+----------+-------+
|      Japan|        20|  21591|
|         US|        16|  20736|
|  Australia|         6|   8211|
|      Italy|         1|   6642|
|     Sweden|         1|   4512|
|     France|         1|   1350|
|Netherlands|         1|   1239|
|    Germany|         1|   1239|
|     Canada|         1|    951|
+-----------+----------+-------+



## Join Types

The `how` argument decides what happens to rows that do **not** find a
match. Two tiny DataFrames make every case visible. Store 3 has no orders,
and order 104 belongs to store 4, which does not exist:

In [4]:
stores = spark.createDataFrame(
    [(1, "Golden Crumbs"), (2, "Tokyo Treats"), (3, "The Crumbly Nook")],
    schema="store_id INT, store_name STRING",
)

orders = spark.createDataFrame(
    [(101, 1, 12), (102, 1, 30), (103, 2, 9), (104, 4, 21)],
    schema="order_id INT, store_id INT, amount INT",
)

The join types follow the SQL `JOIN` clause
([JOIN](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-qry-select-join#parameters));
`how` defaults to `"inner"`:

| `how` | Keeps | Columns from |
| --- | --- | --- |
| `"inner"` (default) | Only matching pairs | Both sides |
| `"left"` | Every left row; `NULL`s where the right side has no match | Both sides |
| `"right"` | Every right row; `NULL`s where the left side has no match | Both sides |
| `"full"` | Every row from both sides | Both sides |
| `"left_semi"` | Left rows that **have** a match | Left side only |
| `"left_anti"` | Left rows that have **no** match | Left side only |
| `"cross"` (or `crossJoin`) | Every combination of rows | Both sides |

:::::{dropdown} Example: How Many Rows Does Each Join Return?
:open:
:color: primary
:icon: pencil
:class-body: example-body

With the 3 `stores` on the left and the 4 `orders` on the right, joined on
`store_id`, count the rows each join type returns.

**Inner.** Store 1 matches two orders and store 2 matches one. Store 3 and
order 104 find no partner and are dropped: $2 + 1 = 3$ rows.

**Left.** The 3 matched rows, plus store 3 with `NULL` order columns:
$3 + 1 = 4$ rows.

**Right.** The 3 matched rows, plus order 104 with a `NULL` store name:
$3 + 1 = 4$ rows.

**Full.** The 3 matched rows, plus both unmatched rows: $3 + 1 + 1 = 5$ rows.

**Left semi.** The stores that have at least one order, each listed **once**
even though store 1 has two orders: 2 rows.

**Left anti.** The stores that have no orders: just store 3, 1 row.

**Cross.** Every store paired with every order: $3 \times 4 = 12$ rows.

The semi and anti joins never duplicate left rows, which is what makes them
the right tool for "which rows have a match?" questions.
:::::

Checking the counts against Spark:

In [5]:
{
    how: stores.join(orders, "store_id", how).count()
    for how in ["inner", "left", "right", "full", "left_semi", "left_anti"]
} | {"cross": stores.crossJoin(orders).count()}

{'inner': 3,
 'left': 4,
 'right': 4,
 'full': 5,
 'left_semi': 2,
 'left_anti': 1,
 'cross': 12}

The full outer join shows both kinds of unmatched rows:

In [6]:
stores.join(orders, "store_id", "full").orderBy("store_id", "order_id").show()

+--------+----------------+--------+------+
|store_id|      store_name|order_id|amount|
+--------+----------------+--------+------+
|       1|   Golden Crumbs|     101|    12|
|       1|   Golden Crumbs|     102|    30|
|       2|    Tokyo Treats|     103|     9|
|       3|The Crumbly Nook|    NULL|  NULL|
|       4|            NULL|     104|    21|
+--------+----------------+--------+------+



### Semi and Anti Joins in Practice

Semi and anti joins answer "which rows have, or lack, a match?" without
adding columns or duplicating rows. Which customers never bought a Tokyo
Tidbits?

In [7]:
tidbits_sales = transactions.filter(F.col("product") == "Tokyo Tidbits")

never_bought_tidbits = customers.join(tidbits_sales, "customerID", "left_anti")

never_bought_tidbits.count()

46

46 of the 300 customers have never bought one. And which franchises have at least one online review?

In [8]:
reviews = spark.read.table("samples.bakehouse.media_customer_reviews")

franchises.join(reviews, "franchiseID", "left_semi").count()

48

All 48 do. A plain inner join would have returned one row per *review*, not per
franchise, and you would have needed a `distinct` afterwards.

:::{admonition} Note: Joins Can Multiply Rows
:class: note

An inner or outer join returns one row for every matching *pair*. If a key
appears $m$ times on the left and $n$ times on the right, the result holds
$m \times n$ rows for that key. Joining two tables on a column that is not
unique on either side can therefore blow up the row count. Check that the
key is unique on at least one side before you join, for example by comparing
`df.count()` with `df.select(key).distinct().count()`.
:::

## Ambiguous Column Names

Franchises and suppliers both have `name`, `city`, `longitude`, `latitude`
and `size` columns. Joining them puts two columns with the same name in the
result, and referring to one of them by name then fails:

In [9]:
franchise_suppliers = franchises.join(suppliers, "supplierID")

franchise_suppliers.select("name").show()

{"ts": "2026-10-12 22:48:15.314", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[AMBIGUOUS_REFERENCE] Reference `name` is ambiguous, could be: [`samples`.`bakehouse`.`sales_franchises`.`name`, `samples`.`bakehouse`.`sales_suppliers`.`name`]. SQLSTATE: 42704\n\nJVM stacktrace:\norg.apache.spark.sql.AnalysisException\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.ambiguousReferenceError(QueryCompilationErrors.scala:4113)\n\tat org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.resolve(package.scala:379)\n\tat org.apache.spark.sql.catalyst.plans.logical.LogicalPlan.resolveChildren(LogicalPlan.scala:295)\n\tat org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anonfun$resolveExpressionByPlanChildren$1(ColumnResolutionHelper.scala:500)\n\tat org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anonfun$resolveExpression$3(ColumnResolutionHelper.scala:170)\n\tat org.apache.spark.sql.catalyst.analysis.package$.withPosition(packa

AnalysisException: [AMBIGUOUS_REFERENCE] Reference `name` is ambiguous, could be: [`samples`.`bakehouse`.`sales_franchises`.`name`, `samples`.`bakehouse`.`sales_suppliers`.`name`]. SQLSTATE: 42704

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.ambiguousReferenceError(QueryCompilationErrors.scala:4113)
	at org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.resolve(package.scala:379)
	at org.apache.spark.sql.catalyst.plans.logical.LogicalPlan.resolveChildren(LogicalPlan.scala:295)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anonfun$resolveExpressionByPlanChildren$1(ColumnResolutionHelper.scala:500)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anonfun$resolveExpression$3(ColumnResolutionHelper.scala:170)
	at org.apache.spark.sql.catalyst.analysis.package$.withPosition(package.scala:110)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anonfun$resolveE

The `AnalysisException` (`AMBIGUOUS_REFERENCE`) lists both candidate
columns. The fix is to give each DataFrame an **alias** and qualify the column
references with it. This is also the natural place to rename the clashing
columns:

In [10]:
f = franchises.alias("f")
s = suppliers.alias("s")

franchise_suppliers = f.join(s, F.col("f.supplierID") == F.col("s.supplierID")).select(
    F.col("f.name").alias("franchise"),
    F.col("f.city").alias("franchise_city"),
    F.col("s.name").alias("supplier"),
    F.col("s.ingredient"),
    F.col("s.approved"),
)

franchise_suppliers.show(5, truncate=False)

+----------------+--------------+------------------+----------+--------+
|franchise       |franchise_city|supplier          |ingredient|approved|
+----------------+--------------+------------------+----------+--------+
|The Crumbly Nook|Sydney        |Cacao Wonders     |cacao     |Y       |
|Tokyo Treats    |Tokyo         |Coconut Grove     |coconut   |Y       |
|Golden Crumbs   |San Francisco |Almond Delights   |almonds   |Y       |
|Sugar Rush      |Melbourne     |Sugar Cane Harvest|cane sugar|Y       |
|Kobe Konfections|Kobe          |Vanilla Valley    |vanilla   |Y       |
+----------------+--------------+------------------+----------+--------+
only showing top 5 rows


Joining on an **expression** like `F.col("f.supplierID") == F.col("s.supplierID")`
keeps *both* key columns, unlike joining on a column name. Using a name, or a
list of names for a composite key, is the simplest way to avoid duplicate
key columns. Use an expression when the key columns have different names or
the condition is not simple equality.

With the aliases in place, a data-quality question becomes easy: are any
franchises supplied by a supplier that is not approved?

In [11]:
franchise_suppliers.filter(F.col("approved") != "Y").show(truncate=False)

+---------+--------------+--------+----------+--------+
|franchise|franchise_city|supplier|ingredient|approved|
+---------+--------------+--------+----------+--------+
+---------+--------------+--------+----------+--------+



The result is empty, so every franchise currently buys from an approved
supplier. An empty result is what a passing data-quality check looks like.

## Join Conditions Beyond Equality

### Self-Joins

A **self-join** joins a DataFrame to itself, for example to find pairs of
franchises in the same city. Aliases are mandatory here, because both sides
have identical column names. The condition `f1.franchiseID < f2.franchiseID`
keeps each pair once and stops a franchise from pairing with itself:

In [12]:
f1 = franchises.alias("f1")
f2 = franchises.alias("f2")

(
    f1.join(
        f2,
        (F.col("f1.city") == F.col("f2.city"))
        & (F.col("f1.franchiseID") < F.col("f2.franchiseID")),
    )
    .select(
        F.col("f1.city").alias("city"),
        F.col("f1.name").alias("franchise_1"),
        F.col("f2.name").alias("franchise_2"),
    )
    .orderBy("city")
    .show(truncate=False)
)

+-------+-----------------+--------------+
|city   |franchise_1      |franchise_2   |
+-------+-----------------+--------------+
|Seattle|Sweet Temptations|Dough Delights|
+-------+-----------------+--------------+



Seattle is the only city with two franchises.

### Range Joins

The condition can be any boolean expression. Suppose the marketing team ran
promotions over a few date ranges, and each sale should be tagged with the
promotion that was running at the time:

In [13]:
promotions = spark.createDataFrame(
    [
        ("Spring Sale", "2024-05-01", "2024-05-03"),
        ("Cookie Week", "2024-05-08", "2024-05-14"),
    ],
    schema="promotion STRING, start_date STRING, end_date STRING",
).select("promotion", F.to_date("start_date").alias("start_date"), F.to_date("end_date").alias("end_date"))

sales_with_promotion = transactions.join(
    promotions,
    F.to_date(transactions.dateTime).between(promotions.start_date, promotions.end_date),
    "left",
)

(
    sales_with_promotion
    .groupBy("promotion")
    .agg(F.count("*").alias("transactions"), F.sum("totalPrice").alias("revenue"))
    .orderBy("promotion")
    .show()
)

+-----------+------------+-------+
|  promotion|transactions|revenue|
+-----------+------------+-------+
|       NULL|        1258|  25611|
|Cookie Week|        1474|  28380|
|Spring Sale|         601|  12480|
+-----------+------------+-------+



The left join keeps sales that happened outside every promotion, grouped
under `NULL`. The Spring Sale revenue of 12480 matches the running total for
3 May in
{doc}`Grouping and Aggregating Data <02_grouping_and_aggregating_data>`, as
it should, because the promotion covers exactly the first three days. Joins without an equality condition cannot be hash-partitioned
on a key, so Spark has to compare many more row pairs. The plan shows it: there
is no hash join, but a nested loop join over the broadcast `promotions`:

In [14]:
sales_with_promotion.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonBroadcastNestedLoopJoin LeftOuter, BuildRight, ((cast(dateTime#23718 as date) >= start_date#23703) AND (cast(dateTime#23718 as date) <= end_date#23704))
         :- PhotonScan parquet samples.bakehouse.sales_transactions[transactionID#23715L,customerID#23716L,franchiseID#23717L,dateTime#23718,product#23719,quantity#23720L,unitPrice#23721L,totalPrice#23722L,paymentMethod#23723,cardNumber#23724L] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://system-tables-prod-us-east-2-uc-metastore-bucket/metastore/b..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<transactionID:bigint,customerID:bigint,franchiseID:bigint,dateTime:timestamp,product:strin..., RequiredDataFilters: []
         +- PhotonShuffleExchangeSource false
            +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#

That is cheap here, because `promotions` is tiny. For a point-in-interval or
interval-overlap condition between larger tables, Databricks has a
**range join optimization**: Databricks SQL applies it automatically, and on
other compute you enable it with a `RANGE_JOIN` hint or a bin-size setting
([Range join optimization](https://docs.databricks.com/aws/en/optimizations/range-join)).

## Null Keys

A comparison with `NULL` returns `NULL`, even `NULL = NULL`, so rows with a
`NULL` key never match in an ordinary join. When `NULL` should match `NULL`,
use the null-safe equality operator `eqNullSafe`, which is `<=>` in SQL and
returns `True` when both operands are `NULL`
([Apache Spark: NULL Semantics](https://spark.apache.org/docs/latest/sql-ref-null-semantics.html)):

In [15]:
left = spark.createDataFrame([("a", 1), (None, 2)], schema="key STRING, left_value INT")
right = spark.createDataFrame([("a", 10), (None, 20)], schema="key STRING, right_value INT")

print("== matches:", left.join(right, left.key == right.key).count())
print("eqNullSafe matches:", left.join(right, left.key.eqNullSafe(right.key)).count())

== matches: 1


eqNullSafe matches: 2


## Broadcast Joins

When one side of a join is small, such as the 48 franchises against all the
transactions, Spark can send a full copy of it to every executor and avoid
shuffling the large side, as explained in
{ref}`Join Strategies <join-strategies>`. Spark does this automatically when
it knows the table is small, and the `broadcast` hint forces it, regardless
of the size threshold
([Hints](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-qry-select-hints#join-hint-types)):

In [16]:
transactions.join(F.broadcast(franchises), "franchiseID").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [franchiseID#23798L, transactionID#23796L, customerID#23797L, dateTime#23799, product#23800, quantity#23801L, unitPrice#23802L, totalPrice#23803L, paymentMethod#23804, cardNumber#23805L, name#23807, city#23808, district#23809, zipcode#23810, country#23811, size#23812, longitude#23813, latitude#23814, supplierID#23815L]
         +- PhotonBroadcastHashJoin [franchiseID#23798L], [franchiseID#23806L], Inner, BuildRight, false, true, false
            :- PhotonScan parquet samples.bakehouse.sales_transactions[transactionID#23796L,customerID#23797L,franchiseID#23798L,dateTime#23799,product#23800,quantity#23801L,unitPrice#23802L,totalPrice#23803L,paymentMethod#23804,cardNumber#23805L] DataFilters: [isnotnull(franchiseID#23798L)], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://system-tables-prod-us-east-2-uc-

The physical plan contains a `PhotonBroadcastHashJoin`, the Photon engine's
version of `BroadcastHashJoin`, instead of a `SortMergeJoin`. `BuildRight`
means the hash table is built from the right side, `franchises`, which is
the side that gets broadcast. Only the small `sales_franchises` scan feeds
an `EXECUTOR_BROADCAST` exchange, and the large transactions scan is not
shuffled at all.

## Set Operations

Set operations combine DataFrames that have the **same columns**, stacking
rows rather than matching them. Note that in SQL, `UNION`, `INTERSECT` and
`EXCEPT` remove duplicates by default
([Set operators](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-qry-select-setops)),
while the DataFrame `union` keeps them:

| Method | Result | SQL equivalent |
| --- | --- | --- |
| `union` / `unionByName` | All rows of both, duplicates kept | `UNION ALL` |
| `intersect` | Distinct rows present in both | `INTERSECT` |
| `intersectAll` | Rows present in both, duplicates kept | `INTERSECT ALL` |
| `subtract` | Distinct rows of the first that are not in the second | `EXCEPT` |
| `exceptAll` | Like `subtract`, duplicates kept | `EXCEPT ALL` |

### `union` Matches Columns by Position

`union` lines up columns by **position**, not by name
([Apache Spark API reference: DataFrame.union](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.union.html)).
If the two DataFrames list the same columns in a different order, the values
silently end up in the wrong columns, or the union fails if the types do not
fit:

In [17]:
may_first_half = spark.createDataFrame([("Pearly Pies", "visa")], schema="product STRING, payment STRING")
may_second_half = spark.createDataFrame([("amex", "Tokyo Tidbits")], schema="payment STRING, product STRING")

may_first_half.union(may_second_half).show()

+-----------+-------------+
|    product|      payment|
+-----------+-------------+
|Pearly Pies|         visa|
|       amex|Tokyo Tidbits|
+-----------+-------------+



`unionByName` matches by column name instead, and with
`allowMissingColumns=True` it fills columns that exist on only one side with
`NULL`
([Apache Spark API reference: DataFrame.unionByName](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.unionByName.html)). Prefer it whenever the inputs come from different places:

In [18]:
may_first_half.unionByName(may_second_half).show()

+-------------+-------+
|      product|payment|
+-------------+-------+
|  Pearly Pies|   visa|
|Tokyo Tidbits|   amex|
+-------------+-------+



In [19]:
with_quantity = spark.createDataFrame([("Orchard Oasis", "visa", 12)], schema="product STRING, payment STRING, quantity INT")

may_first_half.unionByName(with_quantity, allowMissingColumns=True).show()

+-------------+-------+--------+
|      product|payment|quantity|
+-------------+-------+--------+
|  Pearly Pies|   visa|    NULL|
|Orchard Oasis|   visa|      12|
+-------------+-------+--------+



Unlike SQL's `UNION`, `union` does **not** remove duplicates. Call
`distinct()` afterwards when you need set semantics.

### Comparing Sets of Customers

`intersect` and `subtract` compare whole rows, which makes them handy for
comparing lists. Which customers bought from franchises in both the US and
Japan, and which only in the US?

In [20]:
def customers_in(country: str):
    return sales_with_store.filter(F.col("country") == country).select("customerID")


us_customers = customers_in("US")
japan_customers = customers_in("Japan")

print("US:", us_customers.distinct().count())
print("Japan:", japan_customers.distinct().count())
print("Both:", us_customers.intersect(japan_customers).count())
print("US only:", us_customers.subtract(japan_customers).count())

US: 293


Japan: 296


Both: 289


US only: 4


Nearly every customer has bought in both countries. The 289 customers in
both groups plus the 4 US-only customers add up to all 293 US customers,
which is a quick way to check the result.

## Choosing a Join or Set Operation

| Question | Use |
| --- | --- |
| Attach columns from another table | `inner` or `left` join on the key |
| Which rows have a match? | `left_semi` (no extra columns, no duplicated rows) |
| Which rows have no match? | `left_anti` |
| Stack rows from inputs with the same columns | `unionByName` (`union` only when the column order is guaranteed) |
| Rows in both inputs, or only in the first | `intersect`, `subtract` (or the `All` variants to keep duplicates) |
| Join a small lookup table | Broadcast it |
| Match a value to a range | A join on a `between` condition; on large data, the range join optimization |

## Common Mistakes

- **Joining on a key that isn't unique on either side.** Every matching pair
  becomes a row, so the result can be far larger than both inputs. Check key
  uniqueness first, as the note above shows.
- **Selecting an ambiguous column after a join.** Use aliases and qualified
  names, or join on a column name so only one copy of the key remains
  ([Apache Spark API reference: DataFrame.join](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)).
- **Using an inner join plus `distinct` to test for a match.** A `left_semi`
  join returns each left row at most once
  ([JOIN](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-qry-select-join#parameters)).
- **Expecting `NULL` keys to match.** They don't, unless you use
  `eqNullSafe`
  ([Apache Spark: NULL Semantics](https://spark.apache.org/docs/latest/sql-ref-null-semantics.html)).
- **Using `union` on inputs with a different column order.** Values land in
  the wrong columns without an error, as shown above. Use `unionByName`.
- **Expecting the DataFrame `union` to de-duplicate like SQL `UNION`.** It
  keeps duplicates; call `distinct()` afterwards.

Next, {doc}`Working with Complex Data <04_working_with_complex_data>` looks
at columns that hold arrays, structs, maps and semi-structured data.

## Resources

- [`DataFrame.join` (Apache Spark)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)
- [JOIN clause (Apache Spark)](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-join.html)
- [Set operators (Apache Spark)](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-setops.html)
- [Join hints (Apache Spark)](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-hints.html#join-hints)